# 19 — Transformer Encoder in PyTorch

**Learning objective.** Assemble embeddings, positional information, multi-head self-attention and feed-forward blocks using torch.nn.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**token + position embeddings → attention + feed-forward blocks → contextual states → task representation**

Focus on the transformation of information from left to right. Ask what representation changes before asking which library call implements it.

### Reference visual

<p align="center"><img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/The-Transformer-model-architecture.png" width="650" alt="Transformer architecture"/></p>

<sub>Wikimedia Commons — “The-Transformer-model-architecture.png”, CC BY-SA 3.0. Attribution details: `VISUAL_REFERENCES.md`.</sub>

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Increase **number of heads** | representation is split into more attention subspaces | relation diversity can rise but each head gets less width at fixed d_model |
| Increase **depth** | context is repeatedly remixed/transformed | capacity grows with optimization/latency cost |
| Increase sequence length | attention matrix grows quadratically | memory/latency can dominate |

> Before changing a parameter, state the expected direction of the downstream effect.

## Think before running the next cell

1. Why does an encoder need positional information if attention sees all tokens?
2. At fixed model width, what happens to per-head dimension if head count doubles?

### When to use
Use transformer encoders for contextual token/document representations when compute allows.

### When not to use / caution
Do not use a large transformer merely because it is modern; latency/data/interpretability can favor simpler models.

### Debugging lens
Trace tensor shapes through embedding → attention → residual → FFN → residual; shape reasoning catches many bugs.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
import torch, torch.nn as nn
torch.manual_seed(42)
seq_len,batch,d_model=5,2,16
x=torch.randn(batch,seq_len,d_model)
pos=nn.Parameter(torch.randn(1,seq_len,d_model))
layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=4,dim_feedforward=32,batch_first=True,dropout=0.0)
encoder=nn.TransformerEncoder(layer,num_layers=2)
y=encoder(x+pos)
print('input :',tuple(x.shape))
print('output:',tuple(y.shape))
print('same sequence length and model width:',x.shape==y.shape)

input : (2, 5, 16)
output: (2, 5, 16)
same sequence length and model width: True


### What the encoder block adds
1. **Multi-head self-attention** mixes information across token positions.
2. **Feed-forward network** transforms each position independently.
3. **Residual connections + normalization** stabilize deep optimization.
4. **Position information** breaks the permutation symmetry of pure attention.

In [3]:
print(layer)

TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
  )
  (linear1): Linear(in_features=16, out_features=32, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (linear2): Linear(in_features=32, out_features=16, bias=True)
  (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (dropout1): Dropout(p=0.0, inplace=False)
  (dropout2): Dropout(p=0.0, inplace=False)
)


---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Trace transformer tensor shapes
- Explain why positional information is necessary